In [1]:
from inference_sdk import InferenceHTTPClient #getting yolo model from roboflow

CLIENT = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key="AYy5qmSkZoDRX2XHF0Pb"
)

path= "testImages/ap23089090925050_custom-f34f7364319f93a3f76dd247e85c7004b147b31f.jpg"

result = CLIENT.infer(path, model_id="basketball-w2xcw/1")

In [2]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import random

In [3]:
class_colors = {}

def get_class_color(class_name):
    if class_name not in class_colors:
        class_colors[class_name] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
    return class_colors[class_name]

In [4]:
import torch
import cv2
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image

In [5]:
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
import torch.nn as nn

#load HAR MODEL
# weights = R2Plus1D_18_Weights.DEFAULT 
# preprocess = weights.transforms()
# pretrained_model = r2plus1d_18(weights=weights)

# pretrained_model.fc = torch.nn.Identity()

# for param in pretrained_model.parameters(): #unfreeze
#     param.requires_grad = True

# num_classes = 10

# classification_head = torch.nn.Sequential(
#         torch.nn.Linear(512, 256),
#         torch.nn.ReLU(),
#         torch.nn.Dropout(0.5),
#         torch.nn.Linear(256, num_classes),
#     )

# action_model = torch.nn.Sequential(
#     pretrained_model,
#     classification_head
# )

def create_model():
    #load pretrained
    weights = R2Plus1D_18_Weights.DEFAULT
    preprocess = weights.transforms()
    pretrained_model = r2plus1d_18(weights=weights)

    pretrained_model.fc = nn.Identity()

    #custom head
    classification_head = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 10),
    )

    model = nn.Sequential(
        pretrained_model,
        classification_head
    )

    return model

# action_model.load_state_dict(torch.load('modelv2')) #load with weights
# action_model.eval()

#load and build both models
action_model = create_model()
action_modelv2 = create_model()

action_model.load_state_dict(torch.load("model"))
action_modelv2.load_state_dict(torch.load("modelv2"))

action_model.eval()
action_modelv2.eval()

Sequential(
  (0): VideoResNet(
    (stem): R2Plus1dStem(
      (0): Conv3d(3, 45, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
      (1): BatchNorm3d(45, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv3d(45, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
      (4): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Sequential(
          (0): Conv2Plus1D(
            (0): Conv3d(64, 144, kernel_size=(1, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1), bias=False)
            (1): BatchNorm3d(144, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Conv3d(144, 64, kernel_size=(3, 1, 1), stride=(1, 1, 1), padding=(1, 0, 0), bias=False)
          )
          (1): BatchNorm3d(64, eps=1e

In [6]:
import joblib
meta_model = joblib.load("ensemble_model.joblib")

In [7]:
#create transform
transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

#load labels
with open('SpaceJam/dataset/labels_dict.json', 'r') as g:
    labels_wrong = g.read()

data = eval(labels_wrong)

labels = {str(key): i for key, i in data.items()} #converting keys to string rather than integer

labels

{'0': 'block',
 '1': 'pass',
 '2': 'run',
 '3': 'dribble',
 '4': 'shoot',
 '5': 'ball in hand',
 '6': 'defense',
 '7': 'pick',
 '8': 'no_action',
 '9': 'walk',
 '10': 'discard'}

In [8]:
#thread imports
import threading
import queue
import time

In [9]:
from collections import deque

In [10]:
#create tracker objects
trackers = {}
track_history = {}
action_history = {}

#ID counter
object_id_counter = 0

frame_buffers = {} 
last_known_action = {}#

In [11]:
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose_model = mp_pose.Pose()

In [12]:
def initialize_video(input_path, output_path, frame_size=(640, 480), fps=20.0):
    cap = cv2.VideoCapture(input_path)
   # cap.set(cv2.CAP_PROP_FPS, 30)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, frame_size)
    return cap, out

In [13]:
#run object detection
def detect_objects(frame):
    #frame = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    results = CLIENT.infer(frame, model_id="basketball-w2xcw/1")
    return results['predictions'] 

In [24]:
def match_or_create_tracker(frame, bbox): #create object trackers and match
    global object_id_counter

    top_left_x, top_left_y, w, h = bbox
    matched_id = None

    for obj_id, tracker in trackers.items():
        success, tracked_bbox = tracker.update(frame)
        if success:
            tx, ty, tw, th = [int(v) for v in tracked_bbox]
            if abs(tx - top_left_x) < 30 and abs(ty - top_left_y) < 30:  # Small movement threshold
                matched_id = obj_id
                break
    
    if matched_id is None:
        object_id_counter += 1
        matched_id = object_id_counter
        tracker = cv2.legacy.TrackerCSRT_create()
        tracker.init(frame, (top_left_x, top_left_y, w, h))
        trackers[matched_id] = tracker
        action_history[matched_id] = deque(maxlen=3)

    return matched_id

#update trackers
def update_track_history(object_id, x, y):
    if object_id not in track_history:
        track_history[object_id] = []
    track_history[object_id].append((float(x), float(y)))
    if len(track_history[object_id]) > 30:
        track_history[object_id].pop(0)

In [15]:
#HAR single frame verison
# def recognize_action(frame, bbox, object_id):
#     top_left_x, top_left_y, w, h = bbox
#     bottom_right_x, bottom_right_y = top_left_x + w, top_left_y + h

#     pil_image = Image.fromarray(frame)
#     player_crop = pil_image.crop((top_left_x, top_left_y, bottom_right_x, bottom_right_y)).convert("RGB")
#     player_tensor = transform(player_crop).unsqueeze(0).unsqueeze(2)

#     with torch.no_grad():
#         outputs = action_model(player_tensor)
    
#     predicted_action = torch.argmax(outputs, dim=1).item()
#     action_history[object_id].append(predicted_action)

#     last_actions = list(action_history[object_id])
#     return [labels[str(a)] for a in last_actions]

In [14]:
def estimate_pose(frame, bbox):

    top_left_x, top_left_y, w, h = bbox
    bottom_right_x, bottom_right_y = top_left_x + w, top_left_y + h

    #crop image
    person_crop = frame[top_left_y:bottom_right_y, top_left_x:bottom_right_x]
    rgb_frame = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)

    #pose estimate
    results = pose_model.process(rgb_frame)
    
    if results.pose_landmarks:

        for landmark in results.pose_landmarks.landmark:
            cx, cy = int(landmark.x * w) + top_left_x, int(landmark.y * h) + top_left_y
            
            cv2.circle(frame, (cx, cy), 2, (0, 0, 255), -1)  #draw keypoinst

            #show joint connections
            for start_idx, end_idx in mp_pose.POSE_CONNECTIONS:
                start = results.pose_landmarks.landmark[start_idx]
                end = results.pose_landmarks.landmark[end_idx]

                start_x, start_y = int(start.x * w) + top_left_x, int(start.y * h) + top_left_y
                end_x, end_y = int(end.x * w) + top_left_x, int(end.y * h) + top_left_y

                cv2.line(frame, (start_x, start_y), (end_x, end_y), (0,0,0), 2)


    return frame


In [15]:
personLock = threading.Lock()

def proccessPerson(frame, bbox, object_id, action_results, pose_results):
    #create frame buffer for new player if not already present
    if object_id not in frame_buffers:
        frame_buffers[object_id] = []

    #add current frame to the buffer
    frame_buffers[object_id].append((frame, bbox))
    #bbox_sequence = [item[1] for item in frame_buffers[object_id]]

    #buffer full (4 frames), make prediction
    if len(frame_buffers[object_id]) == 4:

        #HAR prediction on frame sequence
        predicted_action = recognize_action_sequence(frame_buffers[object_id])

        #store predicted action
        action_results[object_id] = predicted_action

        #remove top frame from queue
        frame_buffers[object_id].pop()

    #update pose estimation
    pose_frame = estimate_pose(frame, bbox)
    with personLock:
        pose_results[object_id] = pose_frame

In [16]:
import numpy as np
import torch.nn.functional as F

def recognize_action_sequence(frame_bbox_sequence):
    #initalize tensor for frames
    frames_tensor = []

    for frame, bbox in frame_bbox_sequence: #crop to player bbox and add frames to tensor
        top_left_x, top_left_y, w, h = bbox
        bottom_right_x, bottom_right_y = top_left_x + w, top_left_y + h

        pil_image = Image.fromarray(frame)
        crop = pil_image.crop((top_left_x, top_left_y, bottom_right_x, bottom_right_y)).convert("RGB")

        player_tensor = transform(crop)
        frames_tensor.append(player_tensor)


    frames_tensor = torch.stack(frames_tensor)  
    frames_tensor = frames_tensor.permute(1, 0, 2, 3).unsqueeze(0)

    
    # with torch.no_grad():
    #     outputs = action_model(frames_tensor)  
    #     predicted_action = torch.argmax(outputs, dim=1).item()

    # # Return the predicted action label
    # return labels[str(predicted_action)]

    with torch.no_grad():
        probs1 = F.softmax(action_model(frames_tensor), dim=1).cpu().numpy().flatten()
        probs2 = F.softmax(action_modelv2(frames_tensor), dim=1).cpu().numpy().flatten()

    meta_features = np.concatenate([probs1, probs2])
    meta_prediction = meta_model.predict([meta_features])[0]

    return labels[str(meta_prediction)]

In [17]:
def draw_annotations(frame, predictions):
    new_trackers = {}
    bboxDict = {}  # empty dict
    actionDict = {}

    # threading vars
    action_results = {}  #store action results from different threads
    pose_results = {}
    threads = []

    for prediction in predictions:
        x, y, w, h = prediction["x"], prediction["y"], prediction["width"], prediction["height"]
        class_name = prediction["class"]
        color = get_class_color(class_name)

        bbox = (int(x - w / 2), int(y - h / 2), int(w), int(h))
        object_id = match_or_create_tracker(frame, bbox)
        update_track_history(object_id, x, y)

        #draw BBox
        top_left_x, top_left_y, w, h = bbox
        bottom_right_x, bottom_right_y = top_left_x + w, top_left_y + h
        cv2.rectangle(frame, (top_left_x, top_left_y), (bottom_right_x, bottom_right_y), color, 1)

        if class_name == "person":  #proccess person class differently
            thread = threading.Thread(target=proccessPerson, args=(frame, bbox, object_id, action_results, pose_results)) #multithread as this is bottleneck
            thread.start()
            threads.append(thread)

            bboxDict[object_id] = bbox
        
        new_trackers[object_id] = trackers[object_id]

    for thread in threads: #gather proccessPerson threads
        thread.join()

    with personLock:  #shared resource lock
        for object_id, pose_frame in pose_results.items():
            frame = pose_frame

            #if action available then get
            current_action = action_results.get(object_id, "Unknown")

            #use last known action
            if current_action == "Unknown" and object_id in last_known_action:
                current_action = last_known_action[object_id]
            
            #store the last known action if known
            if current_action != "Unknown":
                last_known_action[object_id] = current_action

            #annotate with the current or last known action
            actionDict[object_id] = current_action
            text = f"ID {object_id} Action: {actionDict[object_id]}"
            top_left_x, top_left_y, _, _ = bboxDict[object_id]
            cv2.putText(frame, text, (top_left_x, top_left_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)


    return frame, new_trackers, bboxDict, actionDict

In [18]:
#collision function
def checkCollsion(bbox, otherBBox):
    #print(f"x1 {bbox[0]} y1  {bbox[1]}  x2  {bbox[2]} y2 {bbox[3]}")

    #get width and height
    # bbox_w = bbox[0] - bbox[2]
    # bbox_h = bbox[1] - bbox[3]

    # otherBBox_w = otherBBox[0] - otherBBox[2]
    # otherBBox_h = otherBBox[1] - otherBBox[3]

    #to min max
    x_min, y_min, w, h = bbox
    x_max = x_min + w
    y_max = y_min + h

    other_x_min, other_y_min, other_w, other_h = otherBBox
    other_x_max = other_x_min + other_w
    other_y_max = other_y_min + other_h

    if (x_min < other_x_max and x_max > other_x_min and
        y_min < other_y_max and y_max > other_y_min):
        return True
    else:
        return False

threading

In [19]:
class frameCapture(threading.Thread):
    
    def __init__(self, cap, frame_queue): #construct
        super().__init__()
        self.cap = cap
        self.frame_queue = frame_queue
        self.stopped = False

    def run(self): #add frames to frame queue
        while not self.stopped:
            if not self.frame_queue.full():
                ret, frame = self.cap.read()
                if ret:
                    self.frame_queue.put(frame)
            else:
                time.sleep(0.01)

    def stop(self):
        self.stopped = True


In [20]:
lock = threading.Lock()

class proccessFrame(threading.Thread):

    def __init__(self, frame_queue, proccessed_frame_queue):
        super().__init__()
        self.frame_queue = frame_queue
        self.proccessed_frame_queue = proccessed_frame_queue
        self.stopped = False
    
    def run(self):
        while not self.stopped:
            #if not self.frame_queue.empty():
                try:
                    frame = self.frame_queue.get(timeout=1)
                except queue.Empty:
                    continue

                predictions = detect_objects(frame)

                with lock: #protect due to dictionary issues
                    annotated_frame, updated_trackers, bboxDict, actionDict = draw_annotations(frame, predictions)
                    global trackers
                    trackers = updated_trackers

                    #print(bboxDict)
                    #collision detect
                    for bbox_id, bbox in bboxDict.items():
                    # print(bboxDict)
                        for other_id, otherBBox in bboxDict.items():
                        #    print(2)
                            if bbox_id != other_id and checkCollsion(bbox, otherBBox): #if not same id
                                print(f"collision detected between {bbox_id} ({actionDict[bbox_id]}) and {other_id} ({actionDict[other_id]})")

                                playerA_action = actionDict[bbox_id]
                                playerB_action = actionDict[other_id]

                                #check for potential foul based on action
                                if (playerA_action == "shoot" or playerB_action == "shoot") and (playerA_action == "block" or playerB_action == "block"):
                                    print(f"potential shooting foul found between player {bbox_id} and player {other_id}")

                self.proccessed_frame_queue.put(annotated_frame)

    def stop(self):
        self.stopped = True

In [21]:
class displayFrame(threading.Thread):

    def __init__(self, proccessed_frame_queue, out):
        super().__init__()
        self.proccessed_frame_queue = proccessed_frame_queue
        self.out = out
        self.stopped = False
    
    def run(self):
        while not self.stopped:
            #if not self.proccessed_frame_queue.empty():

                try:
                    annotated_frame = self.proccessed_frame_queue.get(timeout=0.1)
                except queue.Empty:
                    continue

                #annotated_frame = self.proccessed_frame_queue.get()

                self.out.write(annotated_frame)
                cv2.imshow("Tracking", annotated_frame)

                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
                


    def stop(self):
        self.stopped = True


In [22]:
def video_main(input_path, output_path):
    cap, out = initialize_video(input_path, output_path)

    frame_queue = queue.Queue(maxsize=5)
    proccessed_frame_queue = queue.Queue(maxsize=5)

    #create threads
    frame_capture = frameCapture(cap, frame_queue)
    process_threads = [proccessFrame(frame_queue, proccessed_frame_queue) for _ in range(3)]
    frame_display = displayFrame(proccessed_frame_queue, out)


    #start threads
    frame_capture.start()
    
    for thread in process_threads:
        thread.start()

    frame_display.start()

    #quitting
    while True:
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    frame_capture.stop()
    for thread in process_threads:
            thread.stop()
    frame_display.stop()

    frame_capture.join()
    for thread in process_threads:
        thread.join()
    frame_display.join()

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    

In [25]:
videoPath = "JS-ShootingFoul16.mp4"
video_main(videoPath, 'output_video.mp4')

collision detected between 10 (Unknown) and 11 (Unknown)
collision detected between 11 (Unknown) and 10 (Unknown)
collision detected between 6 (Unknown) and 8 (Unknown)
collision detected between 8 (Unknown) and 6 (Unknown)
collision detected between 11 (Unknown) and 10 (Unknown)
collision detected between 10 (Unknown) and 11 (Unknown)
collision detected between 6 (Unknown) and 8 (Unknown)
collision detected between 8 (Unknown) and 6 (Unknown)
collision detected between 11 (Unknown) and 10 (Unknown)
collision detected between 10 (Unknown) and 11 (Unknown)
collision detected between 10 (defense) and 11 (defense)
collision detected between 11 (defense) and 10 (defense)
collision detected between 10 (defense) and 11 (walk)
collision detected between 11 (walk) and 10 (defense)
collision detected between 11 (walk) and 10 (walk)
collision detected between 10 (walk) and 11 (walk)


KeyboardInterrupt: 